# IMPORTACIÓN DE LIBRERIAS

In [1]:
import numpy as np
import pandas as pd
import cma
from scipy.stats import norm
from scipy.stats import qmc
import time
from typing import Callable, List, Tuple
import matplotlib.pyplot as plt
from scipy import stats
import os

# DEFINICIÓN DE LAS CLASES DE CADA SUBPROBLEMA

* `SamplingCMA` -> superclase que se encarga del Sampling, parte común a todos los modelos que compararemos. 
* `GaussianSamplingCMA` -> clase que hereda de `SamplingCMA` que se encarga de aplicar y manejar las excepciones del muestreo Gaussiano Clásico.
* `SobolSamplingCMA` -> clase que hereda de `SamplingCMA` que se encarga de aplicar y manejar las excepciones del muestreo de la secuencia de Sobol.
* `LatinHypercubeSamplingCMA` -> clase que hereda de `SamplingCMA` que se encarga de aplicar y manejar las excepciones del muestreo Latin Hypercube.

In [2]:
class SamplingCMA:
    def __init__(self, dim: int, popsize: int, sigma0: float = 0.2):
        self.dim = dim
        self.popsize = popsize
        self.sigma0 = sigma0
        self.mean = np.zeros(dim)
        self.sigma = sigma0
        self.C = np.eye(dim)
        self.weights = np.log(popsize + 0.5) - np.log(np.arange(1, popsize + 1))
        self.weights = self.weights / np.sum(self.weights)
        self.mu = int(popsize / 2)
        self.weights = self.weights[:self.mu]
        self.mueff = 1 / np.sum(self.weights ** 2)
        self.ps = np.zeros(dim)
        self.pc = np.zeros(dim)
        self.cs = 0.3
        self.c1 = 0.1
        self.cmu = 0.1
        self.damps = 1.0
        self.chiN = np.sqrt(self.dim) * (1 - 1/(4*self.dim) + 1/(21*self.dim**2))
        # Añadir ruido para evitar convergencia prematura
        self.noise_scale = 1e-6
        # Límites para el sigma
        self.sigma_min = 1e-10
        self.sigma_max = 1e10

    def ask(self) -> np.ndarray:
        raise NotImplementedError("Subclasses must implement ask()")

    def tell(self, solutions: np.ndarray, fitness: np.ndarray):
        # Ordenamos segun el fitness
        idx = np.argsort(fitness)
        solutions = solutions[idx]

        # Actualizamos la media
        old_mean = self.mean.copy()
        self.mean = np.sum(self.weights[:, np.newaxis] * solutions[:self.mu], axis=0)

        # Añadimos ruido para evitar convergencias tempranas
        self.mean += np.random.normal(0, self.noise_scale, self.dim)

        # Actualizamos las direciones de evolucion
        y = (self.mean - old_mean) / self.sigma
        self.ps = (1 - self.cs) * self.ps + np.sqrt(self.cs * (2 - self.cs) * self.mueff) * y

        # Actualizamos la matriz de covarianza
        self.C = (1 - self.c1 - self.cmu) * self.C + \
                 self.c1 * np.outer(self.ps, self.ps) + \
                 self.cmu * np.sum(self.weights[:, np.newaxis, np.newaxis] *
                                 np.array([np.outer(s - old_mean, s - old_mean) for s in solutions[:self.mu]]), axis=0)

        # Aseguramos la simetría de la matriz de covarianza
        self.C = (self.C + self.C.T) / 2

        # Añadimos una pequeña regularización a la diagonal
        self.C += 1e-8 * np.eye(self.dim)

        # Actualizamos el tamaño del paso con los límites
        try:
            sigma_update = np.exp((self.cs/self.damps) * (np.linalg.norm(self.ps) / self.chiN - 1))
            self.sigma = np.clip(self.sigma * sigma_update, self.sigma_min, self.sigma_max)
        except:
            # Si hay overflow, mantener el sigma actual
            pass

        # Aseguramos la positividad de la matriz de covarianza
        try:
            np.linalg.cholesky(self.C)
        except np.linalg.LinAlgError:
            # Si no es positiva definida, añadimos regularización
            min_eig = np.min(np.real(np.linalg.eigvals(self.C)))
            if min_eig < 0:
                self.C += (-min_eig + 1e-8) * np.eye(self.dim)

#----------------------------------------------------------------------------------
#----------------------------------------------------------------------------------
#----------------------------------------------------------------------------------

class GaussianSamplingCMA(SamplingCMA):
    def __init__(self, dim: int, popsize: int, sigma0: float = 0.2):
        super().__init__(dim, popsize, sigma0)

    def ask(self) -> np.ndarray:
        solutions = []
        try:
            L = np.linalg.cholesky(self.C)
            for _ in range(self.popsize):
                z = np.random.randn(self.dim)
                x = self.mean + self.sigma * np.dot(L, z)
                solutions.append(x)
        except np.linalg.LinAlgError:
            # añadimos ruido para evitar convergencias tempranas
            for _ in range(self.popsize):
                z = np.random.randn(self.dim)
                x = self.mean + self.sigma * z
                solutions.append(x)
        return np.array(solutions)

#----------------------------------------------------------------------------------
#----------------------------------------------------------------------------------
#----------------------------------------------------------------------------------

class SobolSamplingCMA(SamplingCMA):
    def __init__(self, dim: int, popsize: int, sigma0: float = 0.2):
        super().__init__(dim, popsize, sigma0)
        # Ajustar el tamaño de la población a la potencia de 2 más cercana
        self.sobol_popsize = 2**int(np.ceil(np.log2(popsize)))
        self.sobol = qmc.Sobol(d=dim)
        self.sobol.reset()

    def ask(self) -> np.ndarray:
        solutions = []
        try:
            L = np.linalg.cholesky(self.C)
            sobol_points = self.sobol.random(self.sobol_popsize)
            z = norm.ppf(sobol_points)
            # Seleccionar solo los puntos necesarios
            z = z[:self.popsize]
            for i in range(self.popsize):
                x = self.mean + self.sigma * np.dot(L, z[i])
                solutions.append(x)
        except np.linalg.LinAlgError:
            # añadimos ruido para evitar convergencias tempranas
            sobol_points = self.sobol.random(self.sobol_popsize)
            z = norm.ppf(sobol_points)
            z = z[:self.popsize]
            for i in range(self.popsize):
                x = self.mean + self.sigma * z[i]
                solutions.append(x)
        return np.array(solutions)

#----------------------------------------------------------------------------------
#----------------------------------------------------------------------------------
#----------------------------------------------------------------------------------

class LatinHypercubeSamplingCMA(SamplingCMA):
    def __init__(self, dim: int, popsize: int, sigma0: float = 0.2):
        super().__init__(dim, popsize, sigma0)

    def ask(self) -> np.ndarray:
        solutions = []
        try:
            L = np.linalg.cholesky(self.C)
            sampler = qmc.LatinHypercube(d=self.dim)
            lhc_points = sampler.random(n=self.popsize)
            z = norm.ppf(lhc_points)
            for i in range(self.popsize):
                x = self.mean + self.sigma * np.dot(L, z[i])
                solutions.append(x)
        except np.linalg.LinAlgError:
            # añadimos ruido para evitar convergencias tempranas
            sampler = qmc.LatinHypercube(d=self.dim)
            lhc_points = sampler.random(n=self.popsize)
            z = norm.ppf(lhc_points)
            for i in range(self.popsize):
                x = self.mean + self.sigma * z[i]
                solutions.append(x)
        return np.array(solutions)

# DEFINICIÓN DE LOS PROBLEMAS DE OPTIMIZACIÓN

A continuación, vamos a definir los problemas de optimización con los que vamos a probar la eficacia de cada modelo: 

In [3]:
def sphere(x: np.ndarray) -> float:
    return np.sum(x**2)

def rosenbrock(x: np.ndarray) -> float:
    return np.sum(100 * (x[1:] - x[:-1]**2)**2 + (x[:-1] - 1)**2)

def rastrigin(x: np.ndarray) -> float:
    return 10 * len(x) + np.sum(x**2 - 10 * np.cos(2 * np.pi * x))

def ackley(x: np.ndarray) -> float:
    return -20 * np.exp(-0.2 * np.sqrt(np.mean(x**2))) - np.exp(np.mean(np.cos(2 * np.pi * x))) + 20 + np.e

def schwefel(x: np.ndarray) -> float:
    return 418.9829 * len(x) - np.sum(x * np.sin(np.sqrt(np.abs(x))))

def griewank(x: np.ndarray) -> float:
    sum_sq = np.sum(x**2) / 4000
    prod_cos = np.prod(np.cos(x / np.sqrt(np.arange(1, len(x) + 1))))
    return 1 + sum_sq - prod_cos

def levy(x: np.ndarray) -> float:
    w = 1 + (x - 1) / 4
    term1 = np.sin(np.pi * w[0])**2
    term2 = np.sum((w[:-1] - 1)**2 * (1 + 10 * np.sin(np.pi * w[:-1] + 1)**2))
    term3 = (w[-1] - 1)**2 * (1 + np.sin(2 * np.pi * w[-1])**2)
    return term1 + term2 + term3

def zakharov(x: np.ndarray) -> float:
    sum1 = np.sum(x**2)
    sum2 = np.sum(0.5 * np.arange(1, len(x) + 1) * x)
    return sum1 + sum2**2 + sum2**4

def dixon_price(x: np.ndarray) -> float:
    term1 = (x[0] - 1)**2
    term2 = np.sum(np.arange(2, len(x) + 1) * (2 * x[1:]**2 - x[:-1])**2)
    return term1 + term2

def powell(x: np.ndarray) -> float:
    n = len(x)
    if n % 4 != 0:
        raise ValueError("La dimensión debe ser múltiplo de 4")
    
    result = 0
    for i in range(0, n, 4):
        result += (x[i] + 10*x[i+1])**2 + 5*(x[i+2] - x[i+3])**2 + (x[i+1] - 2*x[i+2])**4 + 10*(x[i] - x[i+3])**4
    return result

def sum_squares(x: np.ndarray) -> float:
    return np.sum(np.arange(1, len(x) + 1) * x**2)

def trid(x: np.ndarray) -> float:
    return np.sum((x - 1)**2) - np.sum(x[1:] * x[:-1])

# EJECUCIÓN DEL CMA-ES

In [4]:
def run_optimization(sampler: SamplingCMA,
                    objective: Callable[[np.ndarray], float],
                    max_iter: int,
                    bounds: Tuple[float, float],
                    initial_point: np.ndarray = None) -> Tuple[List[float], float]:
    best_fitness = float('inf')
    fitness_history = []
    start_time = time.time()
    
    # Inicializar con el punto inicial proporcionado o uno aleatorio
    if initial_point is None:
        initial_point = np.random.uniform(bounds[0], bounds[1], sampler.dim)
    sampler.mean = initial_point.copy()
    
    for _ in range(max_iter):
        solutions = sampler.ask()
        # Guardamos las soluciones dentro de los límites
        solutions = np.clip(solutions, bounds[0], bounds[1])
        fitness = np.array([objective(x) for x in solutions])
        sampler.tell(solutions, fitness)
        
        best_fitness = min(best_fitness, np.min(fitness))
        fitness_history.append(best_fitness)
        
        # Si encontramos una solución óptima, podemos detenernos
        if best_fitness < 1e-10:
            break
    
    end_time = time.time()
    return np.array(fitness_history), end_time - start_time

#----------------------------------------------------------------------------------
#----------------------------------------------------------------------------------
#----------------------------------------------------------------------------------

def plot_time_comparison(results: dict, sampler_dirs: dict):
    """Crea una gráfica comparativa de los tiempos de ejecución para todas las funciones."""
    plt.figure(figsize=(15, 8))
    
    # Preparar datos
    functions = list(results.keys())
    samplers = list(results[functions[0]].keys())
    times = {sampler: [results[func][sampler]['mean_time'] for func in functions] for sampler in samplers}
    
    # Configurar el gráfico de barras
    x = np.arange(len(functions))
    width = 0.25
    colors = ['blue', 'red', 'green']
    
    # Crear barras para cada sampler
    for i, (sampler, color) in enumerate(zip(samplers, colors)):
        plt.bar(x + i*width, times[sampler], width, label=sampler, color=color)
    
    plt.xlabel('Función de Optimización')
    plt.ylabel('Tiempo de Ejecución (segundos)')
    plt.title('Comparación de Tiempos de Ejecución por Método de Muestreo')
    plt.xticks(x + width, functions, rotation=45)
    plt.legend()
    plt.grid(True, axis='y')
    
    # Ajustar el layout para evitar que se corten las etiquetas
    plt.tight_layout()
    
    # Guardar la gráfica
    comparison_dir = os.path.join('imgs', 'comparaciones')
    if not os.path.exists(comparison_dir):
        os.makedirs(comparison_dir)
    
    plt.savefig(os.path.join(comparison_dir, 'tiempo_comparison.png'))
    plt.close()

#----------------------------------------------------------------------------------
#----------------------------------------------------------------------------------
#----------------------------------------------------------------------------------

def plot_comparison(func_name: str, results: dict, sampler_dirs: dict):
    """Crea una gráfica comparativa de los diferentes samplers para una función específica."""
    plt.figure(figsize=(12, 8))
    
    # Obtener los colores para cada sampler
    colors = ['blue', 'red', 'green']
    
    for (sampler_name, stats), color in zip(results.items(), colors):
        mean_fitness = stats['mean_fitness']
        std_fitness = stats['std_fitness']
        
        plt.plot(mean_fitness, label=f'{sampler_name}', color=color)
        plt.fill_between(range(len(mean_fitness)),
                        mean_fitness - std_fitness,
                        mean_fitness + std_fitness,
                        alpha=0.2,
                        color=color)
    
    plt.title(f'{func_name} - Comparison of Sampling Methods')
    plt.xlabel('Iteration')
    plt.ylabel('Best Fitness')
    
    # Solo usar escala logarítmica si todos los valores son positivos
    if all(np.all(stats['mean_fitness'] > 0) for stats in results.values()):
        plt.yscale('log')
    
    plt.legend()
    plt.grid(True)
    
    # Guardar la gráfica comparativa
    comparison_dir = os.path.join('imgs', 'comparaciones')
    if not os.path.exists(comparison_dir):
        os.makedirs(comparison_dir)
    
    plt.savefig(os.path.join(comparison_dir, f'{func_name}_comparison.png'))
    plt.close()

#----------------------------------------------------------------------------------
#----------------------------------------------------------------------------------
#----------------------------------------------------------------------------------

def compare_samplers(dim: int = 10,
                    popsize: int = 50,
                    max_iter: int = 100,
                    n_runs: int = 30,
                    bounds: Tuple[float, float] = (-5, 5)):

    # Ajustar la dimensión para Powell si es necesario
    original_dim = dim
    if dim % 4 != 0:
        dim = (dim // 4) * 4
        print(f"Ajustando dimensión a {dim} para la función Powell")

    # Create directory structure for images
    if not os.path.exists('imgs'):
        os.makedirs('imgs')

    # Create subdirectories for each sampler
    sampler_dirs = {
        'Gaussian': 'imgs/gaussian',
        'Sobol': 'imgs/sobol',
        'Latin Hypercube': 'imgs/latin_hypercube'
    }

    # Create comparison directory
    comparison_dir = os.path.join('imgs', 'comparaciones')
    if not os.path.exists(comparison_dir):
        os.makedirs(comparison_dir)

    for dir_path in sampler_dirs.values():
        if not os.path.exists(dir_path):
            os.makedirs(dir_path)

    # Inicializar samplers con la dimensión ajustada
    samplers = {
        'Gaussian': GaussianSamplingCMA(dim, popsize),
        'Sobol': SobolSamplingCMA(dim, popsize),
        'Latin Hypercube': LatinHypercubeSamplingCMA(dim, popsize)
    }

    functions = {
        'Sphere': sphere,
        'Rosenbrock': rosenbrock,
        'Rastrigin': rastrigin,
        'Ackley': ackley,
        'Schwefel': schwefel,
        'Griewank': griewank,
        'Levy': levy,
        'Zakharov': zakharov,
        'Dixon-Price': dixon_price,
        'Powell': powell,
        'Sum-Squares': sum_squares,
        'Trid': trid
    }

    results = {}

    for func_name, func in functions.items():
        print(f"\nOptimizing {func_name} function...")
        func_results = {}
        all_fitness_by_sampler = {}

        # Generar un punto inicial común para todos los métodos
        initial_point = np.random.uniform(bounds[0], bounds[1], dim)
        print(f"Punto inicial común: {initial_point}")

        for sampler_name, sampler in samplers.items():
            print(f"Running {sampler_name} sampler...")
            all_fitness = []
            all_times = []

            for run in range(n_runs):
                # Reinicializar el sampler para cada ejecución
                sampler = samplers[sampler_name].__class__(dim, popsize)
                # Usar el mismo punto inicial para todas las ejecuciones
                fitness_history, runtime = run_optimization(sampler, func, max_iter, bounds, initial_point)
                all_fitness.append(fitness_history)
                all_times.append(runtime)

            # Convertir a array numpy y asegurar que todas las historias tengan la misma longitud
            max_len = max(len(f) for f in all_fitness)
            padded_fitness = np.array([np.pad(f, (0, max_len - len(f)), mode='edge') for f in all_fitness])
            
            all_fitness_by_sampler[sampler_name] = padded_fitness

            # Calculamos las estadísticas
            mean_fitness = np.mean(padded_fitness, axis=0)
            std_fitness = np.std(padded_fitness, axis=0)
            mean_time = np.mean(all_times)

            func_results[sampler_name] = {
                'mean_fitness': mean_fitness,
                'std_fitness': std_fitness,
                'mean_time': mean_time,
                'initial_point': initial_point
            }

            # Ploteamos la convergencia individual
            plt.figure(figsize=(10, 6))
            plt.plot(mean_fitness, label=f'{sampler_name} (mean)')
            plt.fill_between(range(len(mean_fitness)),
                           mean_fitness - std_fitness,
                           mean_fitness + std_fitness,
                           alpha=0.2)
            plt.title(f'{func_name} - {sampler_name} Convergence\nInitial point: {initial_point}')
            plt.xlabel('Iteration')
            plt.ylabel('Best Fitness')
            
            # Posible uso de escala logarítmica
            if np.all(mean_fitness > 0):
                plt.yscale('log')
            
            plt.legend()
            plt.grid(True)
            
            # guardamos la gráfica en el directorio correspondiente
            plt.savefig(os.path.join(sampler_dirs[sampler_name], f'{func_name}_convergence.png'))
            plt.close()

        results[func_name] = func_results

        # Crear gráfica comparativa para esta función
        plot_comparison(func_name, func_results, sampler_dirs)

        # Comparación estadística
        print(f"\nStatistical comparison for {func_name}:")
        for i, sampler1 in enumerate(samplers.keys()):
            for j, sampler2 in enumerate(samplers.keys()):
                if i < j:
                    # fitness final de cada sampler
                    final_fitness1 = all_fitness_by_sampler[sampler1][:, -1]
                    final_fitness2 = all_fitness_by_sampler[sampler2][:, -1]
                    
                    # ejecutamos si tenemos suficiente datos
                    if len(final_fitness1) > 0 and len(final_fitness2) > 0:
                        stat, p_value = stats.mannwhitneyu(final_fitness1, final_fitness2, alternative='two-sided')
                        print(f"{sampler1} vs {sampler2}: p-value = {p_value:.4f}")
                    else:
                        print(f"{sampler1} vs {sampler2}: Insufficient data for comparison")

    # Crear gráfica comparativa de tiempos
    plot_time_comparison(results, sampler_dirs)

    return results

# EJECUCIÓN

In [14]:
results = compare_samplers(dim=10, popsize=50, max_iter=100, n_runs=30)

print("\nSummary of results:")
for func_name, func_results in results.items():
    print(f"\n{func_name}:")
    for sampler_name, stats in func_results.items():
        print(f"{sampler_name}:")
        print(f"  Final mean fitness: {stats['mean_fitness'][-1]:.6f}")
        print(f"  Mean runtime: {stats['mean_time']:.4f} seconds")

Ajustando dimensión a 8 para la función Powell

Optimizing Sphere function...
Punto inicial común: [ 2.1698426   2.6833701   3.03775529 -1.1457003   3.80137572 -0.97730438
  2.65465643 -1.9554962 ]
Running Gaussian sampler...
Running Sobol sampler...
Running Latin Hypercube sampler...

Statistical comparison for Sphere:
Gaussian vs Sobol: p-value = 0.0000
Gaussian vs Latin Hypercube: p-value = 0.0000
Sobol vs Latin Hypercube: p-value = 0.7845

Optimizing Rosenbrock function...
Punto inicial común: [-0.02685579 -3.65846515 -4.10593406  1.83432872  4.12848758  1.43992901
  3.3545886  -2.13335119]
Running Gaussian sampler...
Running Sobol sampler...
Running Latin Hypercube sampler...

Statistical comparison for Rosenbrock:
Gaussian vs Sobol: p-value = 0.0000
Gaussian vs Latin Hypercube: p-value = 0.0000
Sobol vs Latin Hypercube: p-value = 0.4119

Optimizing Rastrigin function...
Punto inicial común: [ 0.62360632  2.98265524 -2.6266732  -2.02398011 -1.12754563 -2.44264711
  2.63409911 -1.4